## **Atlas Extraction Tutorial**

This tutorial demonstrates how to run LCMV source estimation on EEG data and extract time courses from brain atlases.



In [ ]:
# Install the package (run once)
# !python -m pip install --user --force-reinstall git+https://github.com/cimt-unia/lcmv_xtra.git

#### **Download fsaverage Resources** 


```python
from lcmv_xtra import download_fsaverage

# Define where to store fsaverage resources
FS_DIR = 'path'

# Download fsaverage anatomy and generate required files
download_fsaverage(FS_DIR)
```

> **Note**: Run this only once per project. The same `fsaverage` directory can be reused for all subjects.


In [ ]:
# Libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lcmv_xtra import execute_source_estimation, gt_extraction, difumo_extraction

### **Step 1:** Run LCMV Source Estimation

In [ ]:
# Define your project parameters
FS_DIR = 'fsaverage_path'
PROJECT_ROOT = 'root_path'
SUBJECT_ID = 'sub-01'
TASK = 'rest'
ICA_FILE_PATH = 'fif_path'


# Run source estimation
metadata = execute_source_estimation(
    project_base=PROJECT_ROOT,
    subject_id=SUBJECT_ID,
    task=TASK,
    ica_file_path=ICA_FILE_PATH,
    fsaverage_dir=FS_DIR,
    verbose=True
)

print("✅ Source estimation completed!")
print(f"Output directory: {metadata['subject_output']}")
print(f"Source estimate shape: {metadata['n_sources']} sources × {metadata['n_timepoints']} timepoints")

### **Step 2:** Extract Glasser+Tian Atlas Time Courses

In [ ]:
# Extract time courses from 414-region Glasser+Tian atlas
gt_tc, roi_names = gt_extraction(
    subject_output_dir=metadata['subject_output'],
    global_subjects_dir=metadata['fsaverage_dir'],
    verbose=True
)

print(f"✅ Glasser+Tian extraction completed!")
print(f"Time courses shape: {gt_tc.shape}")  # (414, n_timepoints)
print(f"First 5 ROI names: {roi_names[:5]}")

### **Step 3:** Extract DiFuMo Atlas Time Courses

In [ ]:
# Extract time courses from 512-component DiFuMo atlas
difumo_tc, difumo_info = difumo_extraction(
    subject_output_dir=metadata['subject_output'],
    global_subjects_dir=metadata['fsaverage_dir'],
    verbose=True
)

print(f"✅ DiFuMo extraction completed!")
print(f"Time courses shape: {difumo_tc.shape}")  # (512, n_timepoints)

### **Step 4:** Validate and Visualize Results

In [ ]:
# Check data properties
print("=== DATA VALIDATION ===")
print(f"GT time courses: {gt_tc.shape}, dtype: {gt_tc.dtype}")
print(f"Min: {gt_tc.min():.3f}, Max: {gt_tc.max():.3f}, Mean: {gt_tc.mean():.2e}")
print(f"DiFuMo time courses: {difumo_tc.shape}")

# Plot first 5 regions with actual ROI names
plt.figure(figsize=(15, 10))
for i in range(5):
    plt.subplot(5, 1, i+1)
    plt.plot(gt_tc[i, :1000])  # First 1000 timepoints
    plt.title(f'{roi_names[i]}')
    plt.ylabel('Amplitude')
plt.xlabel('Time')
plt.tight_layout()
plt.show()

### **Step 5:** Save Results for Downstream Analysis

In [ ]:
# Create output filename
output_prefix = f"{metadata['subject_output']}/{metadata['subject_id']}_{metadata['task']}"

# Save time courses
np.save(f"{output_prefix}_gt_timecourses.npy", gt_tc)
np.save(f"{output_prefix}_difumo_timecourses.npy", difumo_tc)

# Save ROI information
pd.DataFrame({'roi_name': roi_names}).to_csv(
    f"{output_prefix}_gt_roi_names.csv", index=False
)
pd.DataFrame(difumo_info).to_csv(
    f"{output_prefix}_difumo_component_info.csv", index=False
)

# Save metadata for reproducibility
import json
with open(f"{output_prefix}_pipeline_metadata.json", 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✅ All results saved to: {metadata['subject_output']}")

### **Step 6:** Optional - Apply Transformations for Analysis

In [ ]:
from sklearn.preprocessing import StandardScaler

# Option 1: Z-score across time (per region) - useful for connectivity analysis
gt_tc_zscore_time = StandardScaler().fit_transform(gt_tc.T).T

# Option 2: Z-score across regions (per timepoint) - useful for machine learning
gt_tc_zscore_regions = StandardScaler().fit_transform(gt_tc)

# Option 3: Baseline correction (first 1000 samples as baseline)
baseline_mean = np.mean(gt_tc[:, :1000], axis=1, keepdims=True)
gt_tc_baseline = gt_tc - baseline_mean

print("Transformations applied successfully!")